<a href="https://colab.research.google.com/github/KaanCesur354/CENG467_TakeHome/blob/main/Ceng467_TakeHomeQ2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Install required libraries
!pip install datasets transformers torch seqeval -q

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Setup done!")
print("GPU available:", torch.cuda.is_available())

Setup done!
GPU available: True


In [2]:
from datasets import load_dataset

dataset = load_dataset("BramVanroy/conll2003")

print(dataset)
print("\n--- Sample training record ---")
print("Tokens:", dataset["train"][0]["tokens"])
print("NER tags:", dataset["train"][0]["ner_tags"])
print("\nLabel list:", dataset["train"].features["ner_tags"].feature.names)

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

--- Sample training record ---
Tokens: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
NER tags: [3, 0, 7, 0, 0, 0, 7, 0, 0]

Label list: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [3]:
# Label mapping
label_list = dataset["train"].features["ner_tags"].feature.names
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for i, l in enumerate(label_list)}
num_labels = len(label_list)

print("Labels:", label_list)
print("Num labels:", num_labels)

# Convert dataset to lists
def extract_data(split):
    tokens = [ex["tokens"] for ex in dataset[split]]
    tags = [ex["ner_tags"] for ex in dataset[split]]
    return tokens, tags

train_tokens, train_tags = extract_data("train")
val_tokens, val_tags = extract_data("validation")
test_tokens, test_tags = extract_data("test")

print(f"\nTrain sentences: {len(train_tokens)}")
print(f"Validation sentences: {len(val_tokens)}")
print(f"Test sentences: {len(test_tokens)}")

# Show BIO tagging example
print("\n--- BIO Tagging Example ---")
for token, tag in zip(train_tokens[0], train_tags[0]):
    print(f"{token:15} {id2label[tag]}")


Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']
Num labels: 9

Train sentences: 14041
Validation sentences: 3250
Test sentences: 3453

--- BIO Tagging Example ---
EU              B-ORG
rejects         O
German          B-MISC
call            O
to              O
boycott         O
British         B-MISC
lamb            O
.               O


In [4]:
!pip install pytorch-crf -q

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchcrf import CRF
from collections import Counter

# Build vocabulary from training tokens
def build_vocab(token_lists, max_vocab=30000):
    counter = Counter()
    for tokens in token_lists:
        counter.update([t.lower() for t in tokens])
    vocab = {"<PAD>": 0, "<UNK>": 1}
    for word, _ in counter.most_common(max_vocab - 2):
        vocab[word] = len(vocab)
    return vocab

vocab = build_vocab(train_tokens)
print(f"Vocabulary size: {len(vocab)}")

# Dataset
class NERDataset(Dataset):
    def __init__(self, token_lists, tag_lists, vocab, max_len=128):
        self.token_lists = token_lists
        self.tag_lists = tag_lists
        self.vocab = vocab
        self.max_len = max_len

    def __len__(self):
        return len(self.token_lists)

    def __getitem__(self, idx):
        tokens = self.token_lists[idx][:self.max_len]
        tags = self.tag_lists[idx][:self.max_len]
        length = len(tokens)

        token_ids = [self.vocab.get(t.lower(), 1) for t in tokens]
        token_ids += [0] * (self.max_len - length)
        tags += [0] * (self.max_len - length)

        return (
            torch.tensor(token_ids, dtype=torch.long),
            torch.tensor(tags, dtype=torch.long),
            torch.tensor(length, dtype=torch.long)
        )

train_dataset = NERDataset(train_tokens, train_tags, vocab)
val_dataset   = NERDataset(val_tokens,   val_tags,   vocab)
test_dataset  = NERDataset(test_tokens,  test_tags,  vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=32)
test_loader  = DataLoader(test_dataset,  batch_size=32)

print("Datasets ready!")

Vocabulary size: 21011
Datasets ready!


In [5]:
# BiLSTM-CRF Model
class BiLSTMCRF(nn.Module):
    def __init__(self, vocab_size, num_labels, embed_dim=128, hidden_dim=256, num_layers=2, dropout=0.3):
        super(BiLSTMCRF, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(
            embed_dim, hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 2, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

    def forward(self, x, tags=None, mask=None):
        embedded = self.dropout(self.embedding(x))
        lstm_out, _ = self.lstm(embedded)
        emissions = self.fc(self.dropout(lstm_out))

        if tags is not None:
            # Training: return negative log likelihood loss
            loss = -self.crf(emissions, tags, mask=mask, reduction='mean')
            return loss
        else:
            # Inference: return best tag sequence
            return self.crf.decode(emissions, mask=mask)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def train_bilstm_crf(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for token_ids, tags, lengths in loader:
        token_ids, tags, lengths = token_ids.to(device), tags.to(device), lengths.to(device)
        mask = torch.arange(token_ids.size(1)).unsqueeze(0).to(device) < lengths.unsqueeze(1)
        optimizer.zero_grad()
        loss = model(token_ids, tags=tags, mask=mask.bool())
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

def evaluate_bilstm_crf(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for token_ids, tags, lengths in loader:
            token_ids, tags, lengths = token_ids.to(device), tags.to(device), lengths.to(device)
            mask = torch.arange(token_ids.size(1)).unsqueeze(0).to(device) < lengths.unsqueeze(1)
            preds = model(token_ids, mask=mask.bool())
            for pred, label, length in zip(preds, tags, lengths):
                l = length.item()
                all_preds.append([id2label[p] for p in pred[:l]])
                all_labels.append([id2label[t.item()] for t in label[:l]])
    return all_preds, all_labels

bilstm_crf = BiLSTMCRF(vocab_size=len(vocab), num_labels=num_labels).to(device)
optimizer_crf = torch.optim.Adam(bilstm_crf.parameters(), lr=1e-3)

EPOCHS = 5
print("Training BiLSTM-CRF...")
for epoch in range(EPOCHS):
    loss = train_bilstm_crf(bilstm_crf, train_loader, optimizer_crf, device)
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {loss:.4f}")

Training BiLSTM-CRF...
Epoch 1/5 | Loss: 7.1432
Epoch 2/5 | Loss: 3.5305
Epoch 3/5 | Loss: 2.4323
Epoch 4/5 | Loss: 1.7997
Epoch 5/5 | Loss: 1.3598


In [6]:
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

val_preds_crf, val_labels_crf = evaluate_bilstm_crf(bilstm_crf, val_loader, device)

print("--- BiLSTM-CRF Results ---")
print(f"Precision: {precision_score(val_labels_crf, val_preds_crf):.4f}")
print(f"Recall:    {recall_score(val_labels_crf, val_preds_crf):.4f}")
print(f"F1-score:  {f1_score(val_labels_crf, val_preds_crf):.4f}")
print("\nDetailed Report:")
print(classification_report(val_labels_crf, val_preds_crf))

--- BiLSTM-CRF Results ---
Precision: 0.8526
Recall:    0.7447
F1-score:  0.7950

Detailed Report:
              precision    recall  f1-score   support

         LOC       0.89      0.85      0.87      1837
        MISC       0.84      0.72      0.77       922
         ORG       0.81      0.64      0.71      1341
         PER       0.85      0.73      0.79      1842

   micro avg       0.85      0.74      0.80      5942
   macro avg       0.85      0.73      0.79      5942
weighted avg       0.85      0.74      0.79      5942


In [7]:
from transformers import BertTokenizerFast, BertForTokenClassification
from torch.optim import AdamW

tokenizer_ner = BertTokenizerFast.from_pretrained("bert-base-cased")

class BertNERDataset(Dataset):
    def __init__(self, token_lists, tag_lists, tokenizer, max_len=128):
        self.token_lists = token_lists
        self.tag_lists = tag_lists
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.token_lists)

    def __getitem__(self, idx):
        tokens = self.token_lists[idx]
        tags = self.tag_lists[idx]
        encoding = self.tokenizer(tokens, is_split_into_words=True, truncation=True, max_length=self.max_len, padding="max_length", return_tensors="pt")
        word_ids = encoding.word_ids()
        aligned_tags = []
        prev_word_id = None
        for word_id in word_ids:
            if word_id is None: aligned_tags.append(-100)
            elif word_id != prev_word_id: aligned_tags.append(tags[word_id])
            else: aligned_tags.append(-100)
            prev_word_id = word_id
        return {"input_ids": encoding["input_ids"].squeeze(), "attention_mask": encoding["attention_mask"].squeeze(), "labels": torch.tensor(aligned_tags, dtype=torch.long)}

bert_ner_train = BertNERDataset(train_tokens, train_tags, tokenizer_ner)
bert_ner_val   = BertNERDataset(val_tokens,   val_tags,   tokenizer_ner)
bert_ner_train_loader = DataLoader(bert_ner_train, batch_size=32, shuffle=True)
bert_ner_val_loader   = DataLoader(bert_ner_val,   batch_size=32)

print("BERT NER Datasets ready!")

BERT NER Datasets ready!


In [8]:
bert_ner_model = BertForTokenClassification.from_pretrained("bert-base-cased", num_labels=num_labels, id2label=id2label, label2id=label2id).to(device)
optimizer_bert_ner = AdamW(bert_ner_model.parameters(), lr=2e-5)

def train_bert_ner(model, loader, optimizer, device):
    model.train()
    total_loss = 0
    for batch in loader:
        input_ids, attention_mask, labels = batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["labels"].to(device)
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        outputs.loss.backward(); optimizer.step()
        total_loss += outputs.loss.item()
    return total_loss / len(loader)

print("Training BERT for NER...")
for epoch in range(3):
    loss = train_bert_ner(bert_ner_model, bert_ner_train_loader, optimizer_bert_ner, device)
    print(f"Epoch {epoch+1}/3 | Loss: {loss:.4f}")

Training BERT for NER...
Epoch 1/3 | Loss: 0.1655
Epoch 2/3 | Loss: 0.0336
Epoch 3/3 | Loss: 0.0192


In [9]:
def evaluate_bert_ner(model, loader, device):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in loader:
            input_ids, attention_mask, labels = batch["input_ids"].to(device), batch["attention_mask"].to(device), batch["labels"].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(-1)
            for pred, label in zip(preds, labels):
                p_tags, t_tags = [], []
                for p, l in zip(pred, label):
                    if l.item() != -100:
                        p_tags.append(id2label[p.item()]); t_tags.append(id2label[l.item()])
                all_preds.append(p_tags); all_labels.append(t_tags)
    return all_preds, all_labels

val_preds_bert, val_labels_bert = evaluate_bert_ner(bert_ner_model, bert_ner_val_loader, device)

print("--- BERT NER Final Summary ---")
print(f"Precision: {precision_score(val_labels_bert, val_preds_bert):.4f}")
print(f"Recall:    {recall_score(val_labels_bert, val_preds_bert):.4f}")
print(f"F1-score:  {f1_score(val_labels_bert, val_preds_bert):.4f}")
print("\nDetailed BERT Report:")
print(classification_report(val_labels_bert, val_preds_bert))

--- BERT NER Final Summary ---
Precision: 0.9334
Recall:    0.9474
F1-score:  0.9404

Detailed BERT Report:
              precision    recall  f1-score   support

         LOC       0.96      0.95      0.96      1837
        MISC       0.87      0.90      0.88       922
         ORG       0.90      0.93      0.91      1341
         PER       0.97      0.98      0.97      1836

   micro avg       0.93      0.95      0.94      5936
   macro avg       0.92      0.94      0.93      5936
weighted avg       0.93      0.95      0.94      5936
